# 🔀 Lab 04 — Put the diff on screen
Run Lab 01 first. The normal bundle run uses `channel_value=tv_app` and should pass. Re-run it with `channel_value=tv-app` to create the deliberate contract failure used for triage. The workspace job can exercise custom instructions. For the PR path, use the separate Git-source job described in the README, pointing to this notebook on a protected base branch in your own fork.

Write down the independent acceptance criteria from Lab 01. Then run the job and record the run ID and source SHA (Git variant). Ask for evidence behind the proposed fix before requesting a review branch.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema');


In [ ]:
%sql
SELECT channel, currency, COUNT(*) AS purchases, SUM(amount_cents) AS cents
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1')
GROUP BY channel, currency ORDER BY channel, currency;


## The contract gate
This cell stands in for the last check before publishing the TV-app total. The normal value is `tv_app`. Pass `tv-app` to reproduce a broken mapping and verify that ZeroOps can investigate the failure. A useful fix restores exactly two unique USD purchases and 2,000 cents while leaving the web purchase out. The assertion is part of the acceptance test. Keep it in the patch.


In [ ]:
%sql
WITH tv_app_purchases AS (
  SELECT purchase_id, currency, amount_cents
  FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1')
  WHERE channel = :channel_value
)
SELECT COUNT(*) AS purchases, SUM(amount_cents) AS cents,
       assert_true(COUNT(*) = 2 AND COUNT(DISTINCT purchase_id) = 2
          AND COUNT_IF(currency = 'USD') = 2 AND SUM(amount_cents) = 2000,
          'TV app contract requires two unique USD purchases and 2000 cents') AS contract_check
FROM tv_app_purchases;


## Take two
For a controlled comparison, use two copies of the job with the same code and parameters so a repeated failure is less likely to reuse the same insight. Investigate one before adding the sample instructions from the README, then configure the instructions and fail the other. Compare the evidence, the size of the proposed change, and whether the report says the instructions were loaded. If a second distinct investigation never appears, record comparison unavailable.

On the Git path, inspect any candidate diff and run a separate scratch job against the candidate commit. Compare it with the original failing commit. If pushing is unavailable, review a manually prepared patch. Track any generated PR, successful rerun, and resolution as separate events.
